In [78]:
import numpy as np
import pandas as pd
import ast as ast
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.metrics.pairwise import cosine_similarity

Importing the dataset

In [ ]:
movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

In [80]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,imdb_id
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,10/12/2009,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,499549.0
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,19/05/2007,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,449088.0


In [ ]:
credits.head()

In [89]:
movies = movies.merge(credits, on = 'title')
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title,vote_average,vote_count,imdb_id,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,499549.0,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,449088.0,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [90]:
#checking the dustribution of movie language

movies['original_language'].value_counts()
# movies.isna().sum()
# movies['imdb'].info()

en    4509
fr      70
es      32
zh      27
de      27
hi      19
ja      16
it      14
ko      12
cn      12
ru      11
pt       9
da       7
sv       5
nl       4
fa       4
th       3
he       3
ta       2
cs       2
ro       2
id       2
ar       2
vi       1
sl       1
ps       1
no       1
ky       1
hu       1
pl       1
af       1
nb       1
tr       1
is       1
xx       1
te       1
el       1
Name: original_language, dtype: int64

In [91]:
#genre
#id
#keywords
#title
#overview
#cast
#crew

movies = movies[['movie_id','imdb_id', 'genres','keywords', 'title', 'overview', 'cast', 'crew']]

In [92]:
#checking null values

movies.isnull().sum()

movie_id      0
imdb_id     208
genres        0
keywords      0
title         0
overview      3
cast          0
crew          0
dtype: int64

In [93]:
#removing null values, as overview is key factor

movies.dropna(inplace = True)
movies['imdb_id'] = movies['imdb_id'].astype('int64')
movies.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 4599 entries, 0 to 4807
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4599 non-null   int64 
 1   imdb_id   4599 non-null   int64 
 2   genres    4599 non-null   object
 3   keywords  4599 non-null   object
 4   title     4599 non-null   object
 5   overview  4599 non-null   object
 6   cast      4599 non-null   object
 7   crew      4599 non-null   object
dtypes: int64(2), object(6)
memory usage: 323.4+ KB


In [94]:
#converting Imdb rating

def convert_imdb_id(x):
    x = str(int(x))

    if len(x) == 6:
        return 'tt0' + x
    elif len(x) == 7:
        return 'tt' + x

movies['imdb_id'] = movies['imdb_id'].apply(convert_imdb_id)
movies['imdb_id'] = movies['imdb_id'].astype('string')
movies.info()



<class 'pandas.core.frame.DataFrame'>
Int64Index: 4599 entries, 0 to 4807
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4599 non-null   int64 
 1   imdb_id   4058 non-null   string
 2   genres    4599 non-null   object
 3   keywords  4599 non-null   object
 4   title     4599 non-null   object
 5   overview  4599 non-null   object
 6   cast      4599 non-null   object
 7   crew      4599 non-null   object
dtypes: int64(1), object(6), string(1)
memory usage: 323.4+ KB


In [95]:
movies.tail(10)

,movie_id,imdb_id,genres,keywords,title,overview,cast,crew
4794,39851,tt0388838,"[{""id"": 18, ""name"": ""Drama""}]","[{""id"": 6782, ""name"": ""addiction""}, {""id"": 155...",Clean,"After losing her husband to a heroin overdose,...","[{""cast_id"": 1, ""character"": ""Emily Wang"", ""cr...","[{""credit_id"": ""52fe47369251416c9106db9b"", ""de..."
4795,13898,tt0255094,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10769, ""n...",[],The Circle,Various women struggle to function in the oppr...,"[{""cast_id"": 3, ""character"": ""Nargess"", ""credi...","[{""credit_id"": ""52fe45b09251416c7505f189"", ""de..."
4797,36095,tt0123948,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 27, ""name...","[{""id"": 233, ""name"": ""japan""}, {""id"": 549, ""na...",Cure,A wave of gruesome murders is sweeping Tokyo. ...,"[{""cast_id"": 3, ""character"": ""Kenichi Takabe"",...","[{""credit_id"": ""52fe45cc9251416c9103eb7b"", ""de..."
4800,124606,tt0109266,"[{""id"": 18, ""name"": ""Drama""}]","[{""id"": 10726, ""name"": ""gang""}, {""id"": 33928, ...",Bang,A young woman in L.A. is having a bad day: she...,"[{""cast_id"": 2, ""character"": ""The Girl"", ""cred...","[{""credit_id"": ""52fe4ab0c3a368484e161add"", ""de..."
4801,14337,tt0390384,"[{""id"": 878, ""name"": ""Science Fiction""}, {""id""...","[{""id"": 1448, ""name"": ""distrust""}, {""id"": 2101...",Primer,Friends/fledgling entrepreneurs invent a devic...,"[{""cast_id"": 1, ""character"": ""Aaron"", ""credit_...","[{""credit_id"": ""52fe45e79251416c75066791"", ""de..."
4803,9367,tt0104815,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 5616, ""name"": ""united states\u2013mexi...",El Mariachi,El Mariachi just wants to play his guitar and ...,"[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de..."
4804,72766,tt1880418,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",[],Newlyweds,A newlywed couple's honeymoon is upended by th...,"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de..."
4805,231617,tt3000844,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...","Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de..."
4806,126186,tt2070597,[],[],Shanghai Calling,When ambitious New York attorney Sam is sent t...,"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id...","[{""credit_id"": ""52fe4ad9c3a368484e16a36b"", ""de..."
4807,25975,tt0378407,"[{""id"": 99, ""name"": ""Documentary""}]","[{""id"": 1523, ""name"": ""obsession""}, {""id"": 224...",My Date with Drew,Ever since the second grade when he first saw ...,"[{""cast_id"": 3, ""character"": ""Herself"", ""credi...","[{""credit_id"": ""58ce021b9251415a390165d9"", ""de..."


In [96]:
#checking duplicate values

movies.duplicated().sum() 

#result - no duplicated values found

0

Preprocessing Genres Column

In [97]:
#evaluating genres column 
movies['genres'].iloc[0]
# type(movies['genres'])
#genres column is a string

#creating a function for fetch only movie genres list

def convert(genres):
    L = []
    for i in ast.literal_eval(genres):
        L.append(i['name'])
    return L


movies['genres'] = movies['genres'].apply(convert)

In [98]:
#applying the same function on keywords column

movies['keywords'] = movies['keywords'].apply(convert)
movies.head(5)


,movie_id,imdb_id,genres,keywords,title,overview,cast,crew
0,19995,tt0499549,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,tt0449088,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,tt2379713,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",Spectre,A cryptic message from Bond’s past sends him o...,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,tt1345836,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,tt0401729,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",John Carter,"John Carter is a war-weary, former military ca...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [99]:
#cleaning cast column

# movies['cast'].iloc[0]

def convert_cast(cast):
    L = []
    for i in ast.literal_eval(cast):
        L.append(i['character'])
    return L

movies['cast'] = movies['cast'].apply(convert)

movies['cast'] = movies['cast'].apply(lambda x: x[:3])
print(movies['cast'])

0        [Sam Worthington, Zoe Saldana, Sigourney Weaver]
1           [Johnny Depp, Orlando Bloom, Keira Knightley]
2            [Daniel Craig, Christoph Waltz, Léa Seydoux]
3            [Christian Bale, Michael Caine, Gary Oldman]
4          [Taylor Kitsch, Lynn Collins, Samantha Morton]
                              ...                        
4803    [Carlos Gallardo, Jaime de Hoyos, Peter Marqua...
4804         [Edward Burns, Kerry Bishé, Marsha Dietlein]
4805           [Eric Mabius, Kristin Booth, Crystal Lowe]
4806            [Daniel Henney, Eliza Coupe, Bill Paxton]
4807    [Drew Barrymore, Brian Herzlinger, Corey Feldman]
Name: cast, Length: 4599, dtype: object


In [100]:
movies['crew'].iloc[0]

def get_director(crew):
    L = []
    for i in ast.literal_eval(crew):
        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

movies['crew'] = movies['crew'].apply(get_director)
movies['crew'].iloc[0]

['James Cameron']

In [101]:
movies.head(3)

,movie_id,imdb_id,genres,keywords,title,overview,cast,crew
0,19995,tt0499549,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",Avatar,"In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,tt0449088,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,tt2379713,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",Spectre,A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]


In [102]:
#converting overview column into a list

movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['overview'].iloc[0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.']

In [103]:
movies.head(2)

,movie_id,imdb_id,genres,keywords,title,overview,cast,crew
0,19995,tt0499549,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,tt0449088,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]


In [104]:
movies[movies['crew'].isna()].count()


movie_id    0
imdb_id     0
genres      0
keywords    0
title       0
overview    0
cast        0
crew        0
dtype: int64

In [105]:
#removing space between cast and crew names

movies['overview'] = movies['overview'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])


In [106]:
movies.head(2)

,movie_id,imdb_id,genres,keywords,title,overview,cast,crew
0,19995,tt0499549,"[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,tt0449088,"[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]


In [107]:
#Concating all columns to convert them into tags

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

movies.head(4)

,movie_id,imdb_id,genres,keywords,title,overview,cast,crew,tags
0,19995,tt0499549,"[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...",Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,tt0449088,"[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...",Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,tt2379713,"[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...",Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,tt1345836,"[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...",The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."


In [108]:
#dropping additional columns

new_df = movies[['movie_id', 'imdb_id', 'title', 'tags']]


In [109]:
new_df

,movie_id,imdb_id,title,tags
0,19995,tt0499549,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,tt0449088,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,tt2379713,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,tt1345836,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,tt0401729,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...,...
4803,9367,tt0104815,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui..."
4804,72766,tt1880418,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended..."
4805,231617,tt3000844,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,..."
4806,126186,tt2070597,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is..."


In [110]:
#converting into text/string

new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

C:\Users\sshuk\AppData\Local\Temp\ipykernel_3452\741588699.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [111]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\sshuk\AppData\Local\Temp\ipykernel_3452\1380776331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [112]:
new_df['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [113]:
new_df['tags'][1]

"captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. adventure fantasy action ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife traitor shipwreck strongwoman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger johnnydepp orlandobloom keiraknightley goreverbinski"

Text Vectorisation Using Bag of Words Technique

In [114]:
#stemming using nltk library

ps = PorterStemmer()

In [115]:
#Creating a function to stem the tags column in new_df

def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [116]:
new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\sshuk\AppData\Local\Temp\ipykernel_3452\3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [117]:
cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

In [118]:
feature_names = cv.get_feature_names_out().tolist()
feature_names

['000',
 '007',
 '10',
 '100',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '17th',
 '18',
 '18th',
 '18thcenturi',
 '19',
 '1910',
 '1920',
 '1930',
 '1940',
 '1944',
 '1950',
 '1950s',
 '1960',
 '1960s',
 '1970',
 '1970s',
 '1976',
 '1980',
 '1985',
 '1990',
 '1995',
 '1999',
 '19th',
 '19thcenturi',
 '20',
 '200',
 '2003',
 '2009',
 '20th',
 '21st',
 '23',
 '24',
 '25',
 '30',
 '300',
 '3d',
 '40',
 '50',
 '500',
 '60',
 '70',
 'aaron',
 'aaroneckhart',
 'abandon',
 'abduct',
 'abigailbreslin',
 'abil',
 'abl',
 'aboard',
 'abov',
 'abus',
 'academ',
 'academi',
 'accept',
 'access',
 'accid',
 'accident',
 'acclaim',
 'accompani',
 'accomplish',
 'account',
 'accus',
 'ace',
 'achiev',
 'acquaint',
 'act',
 'action',
 'actionhero',
 'activ',
 'activist',
 'activities',
 'actor',
 'actress',
 'actual',
 'adam',
 'adamsandl',
 'adamshankman',
 'adapt',
 'add',
 'addict',
 'adjust',
 'admir',
 'admit',
 'adolesc',
 'adopt',
 'ador',
 'adrienbrodi',
 'adult',
 'adultanim',
 'adult

Calculate Cosine distance between two vectors

In [33]:
similarity = cosine_similarity(vectors)

Creating the Recommender System

In [119]:
#Sorting the array based on the Similarity Score

top_movies = sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x: x[1])[1:6]
top_movies

[(1216, 0.28676966733820225),
 (2409, 0.26901379342448517),
 (3730, 0.2605130246476754),
 (507, 0.255608593705383),
 (539, 0.2503866978335957)]

In [120]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    top_movies = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    for i in top_movies:
        print(new_df.iloc[i[0]].title)


In [121]:
recommend('The Avengers')

Iron Man 3
Avengers: Age of Ultron
Captain America: Civil War
Captain America: The First Avenger
Iron Man


In [122]:
import pickle

In [123]:
##Creating pickle file for new_df dataframe

pickle.dump(new_df.to_dict(), open('movies.pkl','wb'))


In [124]:
##Creating pickle file for similarity array

pickle.dump(similarity, open('similarity.pkl','wb'))